# 7T LCModel metabolite-map comparison

Publication-style comparison of No suppression, WALINET style, U20, and the final WALRUS model. The high-resolution anatomy is aligned to the MRSI geometry. For each metabolite, the color scale is determined from the final WALRUS map and then applied unchanged to both slices and all four methods.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path('.').resolve()

SUBJECT = Path('UCSF/Vol1')
SUPERIOR_SLICE = 17  # A, MRSI z-index
INFERIOR_SLICE = 5  # B, MRSI z-index

PIPELINES = [
    ('No suppression', 'LCModelB4Walrus'),
    ('shallow U-Net \n (heuristic training)', 'LCModelU20_WALINET_Calibration_S1'),
    ('shallow U-Net \n (calibrated training)', 'LCModelU20_S1'),
    ('shallow Y-Net + L2 prior \n (calibrated training)', 'LCModelY16_S1'),
    ('deep U-Net \n (calibrated)', 'LCModelAfterWalrus'),
    
   # ('U-Net\n(heuristic training)', 'LCModelU20_WALINET_Calibration_S1'),
   # ('U-Net\n(calibrated training)', 'LCModelU20_S1'),
]
METABOLITES = {
    'tNAA': 'NAA_NAAG',
    'tCr':  'Cr_PCr',
    'tCho': 'GPC_PCh',
    'Glx':  'Glu_Gln',
}

VMAX_PERCENTILE = 97.5  # aus der finalen WALRUS-Karte bestimmt
VMAX_REFERENCE_PIPELINE = 'LCModelAfterWalrus'
ANATOMY_UPSAMPLING = 4
MAP_ALPHA = 1  # identische Darstellung wie in der 3T-Figure
LOCALIZER_ORIENTATION = 'sagittal'  # 'coronal' oder 'sagittal'
FONT_SCALE = 1.15  # leicht größere Beschriftungen für LaTeX
COLORMAP = 'viridis'
FIGSIZE = (12.0, 14.7)  # stacked slices: Slice A block above Slice B block
STACK_SLICES = True

SAVE_PDF = "SavedGraphics/7T_metabolite_maps.pdf"  # z.B. PROJECT_ROOT / 'results/PaperFigures/7T_metabolite_maps.pdf'
SELECTION_FILE = PROJECT_ROOT / 'configs/PaperSpectra/7T_Vol3_UCSF.yaml'
SHOW_SELECTED_VOXELS = False

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.metabolite_map_figure import create_figure
from walinet.paper_figures.spectrum_selection import load_selected_voxels

b0_dir = PROJECT_ROOT / 'data/7T/B0corrected_wo_LipidMask' / SUBJECT
no_b0_dir = PROJECT_ROOT / 'data/7T/NoB0Correction' / SUBJECT
selected_voxels = (
    load_selected_voxels(SELECTION_FILE)
    if SHOW_SELECTED_VOXELS and SELECTION_FILE.is_file()
    else {}
)

figure, qc = create_figure(
    b0_subject_dir=b0_dir,
    no_b0_subject_dir=no_b0_dir,
    pipelines=PIPELINES,
    metabolites=METABOLITES,
    superior_slice=SUPERIOR_SLICE,
    inferior_slice=INFERIOR_SLICE,
    vmax_percentile=VMAX_PERCENTILE,
    vmax_reference_pipeline=VMAX_REFERENCE_PIPELINE,
    anatomy_upsampling=ANATOMY_UPSAMPLING,
    cmap=COLORMAP,
    map_alpha=MAP_ALPHA,
    localizer_orientation=LOCALIZER_ORIENTATION,
    font_scale=FONT_SCALE,
    figsize=FIGSIZE,
    selected_voxels=selected_voxels,
    stack_slices=STACK_SLICES,
)

if SAVE_PDF is not None:
    save_path = Path(SAVE_PDF).expanduser()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f'Saved PDF: {save_path.resolve()}')

print(f"MRSI grid: {qc['mask_shape']}")
print(f"Original anatomy: {qc['anatomy_shape_original']}")
print(f"MRSI-aligned high-resolution anatomy: {qc['anatomy_shape_resampled']}")
print(f"Map orientation transform: {qc['array_orientation_transform']} "
      f"(mask Dice={qc['array_orientation_dice']:.5f})")

## Full-length No suppression vs WALRUS, compact map panel

In [ ]:
from walinet.paper_figures.metabolite_map_figure import create_two_method_full_length_figure

FULL_LENGTH_PIPELINES = [
    ('No suppression', 'LCModelB4Walrus'),
    ('WALRUS', 'LCModelAfterWalrus'),
]
FULL_LENGTH_METABOLITES = {
    'tNAA': 'NAA_NAAG',
    'Glx': 'Glu_Gln',
}
FULL_LENGTH_SAVE_PDF = Path('SavedGraphics/7T_metabolite_maps_full_length_no_suppression_vs_walrus.pdf')

full_length_figure, full_length_qc = create_two_method_full_length_figure(
    b0_subject_dir=b0_dir,
    no_b0_subject_dir=no_b0_dir,
    pipelines=FULL_LENGTH_PIPELINES,
    metabolites=FULL_LENGTH_METABOLITES,
    superior_slice=SUPERIOR_SLICE,
    inferior_slice=INFERIOR_SLICE,
    vmax_percentile=VMAX_PERCENTILE,
    vmax_reference_pipeline='LCModelAfterWalrus',
    anatomy_upsampling=ANATOMY_UPSAMPLING,
    cmap=COLORMAP,
    map_alpha=MAP_ALPHA,
    localizer_orientation=LOCALIZER_ORIENTATION,
    font_scale=FONT_SCALE,
    figsize=(16.5, 3.1),
)

FULL_LENGTH_SAVE_PDF.parent.mkdir(parents=True, exist_ok=True)
full_length_figure.savefig(FULL_LENGTH_SAVE_PDF, format='pdf', bbox_inches='tight')
print(f'Saved PDF: {FULL_LENGTH_SAVE_PDF.resolve()}')
print(f"Map orientation transform: {full_length_qc['array_orientation_transform']} "
      f"(mask Dice={full_length_qc['array_orientation_dice']:.5f})")
